# 🎨 WikiArt Caption Viewer

`generate_wikiart_captions.py` で生成した BLIP キャプションと、  
元の WikiArt 画像・メタデータを並べて確認するノートブックです。

---
**表示レイアウト（1 セルあたり）**

```
┌────────────────────────────────┐
│          画像 (PIL)             │
├────────────────────────────────┤
│ ✅ BLIP キャプション (緑)       │
│ 🏷  Style / Genre / Artist (青) │
│ ⚠️  元テンプレートテキスト (グレー) │
└────────────────────────────────┘
```


## ① 設定

In [ ]:
# ============================================================
# 設定セル — ここだけ変更すれば OK
# ============================================================

# generate_wikiart_captions.py が出力した Parquet ファイルのパス
CAPTIONS_PATH = "./wikiart_captions_out/wikiart_blip_captions.parquet"

# HuggingFace キャッシュディレクトリ（None で デフォルト）
HF_CACHE_DIR  = None

# ── 表示設定 ──────────────────────────────────────────────────
N_SAMPLES = 12     # 一度に表示する件数
NCOLS     = 4      # グリッドの列数
SEED      = 42     # 乱数シード（変えると別サンプルになる）

# ── フィルタ（不要なら None に） ─────────────────────────────
STYLE_FILTER  = None   # 例: "Impressionism"
GENRE_FILTER  = None   # 例: "landscape"

# ── 画像サイズ（インチ） ──────────────────────────────────────
CELL_W = 3.5           # 1 セルの幅
CELL_H = 5.2           # 1 セルの高さ

print("設定完了 ✓")


## ② ライブラリの読み込み・ユーティリティ関数

In [ ]:
import random
import textwrap
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd
from PIL import Image

# インライン表示
%matplotlib inline
matplotlib.rcParams["figure.dpi"] = 120

print("ライブラリ読み込み完了 ✓")


In [ ]:
# ──────────────────────────────────────────────────────────────
# ユーティリティ関数
# ──────────────────────────────────────────────────────────────

def get_meta(row: dict, col_map: dict) -> str:
    """スタイル / ジャンル / 作者のメタデータ文字列を組み立てる。"""
    parts = []
    for label, key in [("Style", "style"), ("Genre", "genre"), ("Artist", "artist")]:
        col = col_map.get(key)
        if col and row.get(col):
            parts.append(f"{label}: {row[col]}")
    return " | ".join(parts)


def get_generated_caption(df: pd.DataFrame, image_idx: int) -> str:
    """Parquet から生成キャプションを取得する。"""
    rows = df[df["image_idx"] == image_idx]
    return str(rows.iloc[0]["caption"]) if not rows.empty else "(未生成)"


def get_orig_text(row: dict) -> str:
    """
    元データセットのテキスト列（テンプレートキャプション）を返す。
    List[str] の場合は最初の 1 件だけ使用する（比較用なので簡潔に）。
    """
    raw = row.get("text", "")
    if isinstance(raw, list):
        return raw[0] if raw else ""
    return str(raw)


def wrap(text: str, width: int = 42) -> str:
    """テキストを指定幅で折り返す。"""
    return "\n".join(textwrap.wrap(text, width)) if text else ""


def sample_indices(
    df: pd.DataFrame,
    hf_ds,
    col_map: dict,
    n: int,
    seed: int,
    style_filter=None,
    genre_filter=None,
):
    """フィルタ条件に合う image_idx をランダムサンプリングする。"""
    candidates = df["image_idx"].tolist()

    if style_filter or genre_filter:
        filtered = []
        for idx in candidates:
            row = hf_ds[int(idx)]
            if style_filter and row.get(col_map.get("style"), "") != style_filter:
                continue
            if genre_filter and row.get(col_map.get("genre"), "") != genre_filter:
                continue
            filtered.append(idx)
        candidates = filtered
        print(f"フィルタ後の候補件数: {len(candidates):,}")

    if not candidates:
        raise ValueError("フィルタ条件に合う画像が見つかりませんでした。")

    random.seed(seed)
    n = min(n, len(candidates))
    return random.sample(candidates, n)


print("ユーティリティ関数定義完了 ✓")


## ③ データの読み込み

In [ ]:
# ── 生成キャプション（Parquet）──────────────────────────────
captions_path = Path(CAPTIONS_PATH)
assert captions_path.exists(), (
    f"ファイルが見つかりません: {captions_path}\n"
    "先に generate_wikiart_captions.py を実行してください。"
)

df_caps = pd.read_parquet(captions_path)
print(f"生成キャプション: {len(df_caps):,} 件")
df_caps.head(3)


In [ ]:
# ── HuggingFace データセット ────────────────────────────────
from datasets import load_dataset as hf_load

hf_ds = hf_load("fusing/wikiart_captions", split="train", cache_dir=HF_CACHE_DIR)
print(f"WikiArt 画像数: {len(hf_ds):,} 件")
print(f"カラム: {hf_ds.column_names}")

# スタイル・ジャンル・作者の列名を自動検出
cols = hf_ds.column_names
col_map = {
    "style":  next((c for c in ["style",  "art_style", "Style"]  if c in cols), None),
    "genre":  next((c for c in ["genre",  "Genre"]               if c in cols), None),
    "artist": next((c for c in ["artist", "Artist", "author"]    if c in cols), None),
}
print(f"\n検出された列: {col_map}")


In [ ]:
# 利用可能なスタイル / ジャンル の一覧を確認（フィルタ設定の参考に）
if col_map["style"]:
    styles = sorted(set(hf_ds[col_map["style"]]))
    print(f"スタイル一覧 ({len(styles)} 種):")
    print(" / ".join(styles[:20]), "..." if len(styles) > 20 else "")

if col_map["genre"]:
    genres = sorted(set(hf_ds[col_map["genre"]]))
    print(f"\nジャンル一覧 ({len(genres)} 種):")
    print(" / ".join(genres[:20]), "..." if len(genres) > 20 else "")


## ④ グリッド表示

`SEED` を変えると別のサンプルが表示されます。  
`STYLE_FILTER` / `GENRE_FILTER` で特定カテゴリに絞り込めます。


In [ ]:
def show_grid(
    df:           pd.DataFrame,
    hf_ds,
    col_map:      dict,
    n_samples:    int   = N_SAMPLES,
    ncols:        int   = NCOLS,
    seed:         int   = SEED,
    style_filter        = STYLE_FILTER,
    genre_filter        = GENRE_FILTER,
    cell_w:       float = CELL_W,
    cell_h:       float = CELL_H,
    save_path:    str   = None,      # 保存先（None で保存しない）
):
    """
    ランダムサンプリングした画像とキャプションをグリッド表示する。

    Parameters
    ----------
    save_path : str or None
        指定すると PNG として保存する（例: "./review.png"）
    """
    # ── サンプリング ───────────────────────────────────────────
    indices = sample_indices(
        df, hf_ds, col_map, n_samples, seed, style_filter, genre_filter
    )
    n    = len(indices)
    nrows = (n + ncols - 1) // ncols

    # ── Figure 作成 ───────────────────────────────────────────
    fig_w = cell_w * ncols
    fig_h = cell_h * nrows
    fig   = plt.figure(figsize=(fig_w, fig_h), facecolor="#1e1e2e")

    outer = gridspec.GridSpec(
        nrows, ncols, figure=fig,
        hspace=0.05, wspace=0.03,
        left=0.01, right=0.99,
        top=0.96,  bottom=0.01,
    )

    # ── 各セルを描画 ──────────────────────────────────────────
    for pos, idx in enumerate(indices):
        r, c = divmod(pos, ncols)

        # 上（画像）: 下（テキスト）= 3 : 2 に分割
        inner = gridspec.GridSpecFromSubplotSpec(
            2, 1,
            subplot_spec=outer[r, c],
            height_ratios=[3, 2],
            hspace=0.0,
        )

        # ── 画像エリア ──────────────────────────────────────
        ax_img = fig.add_subplot(inner[0])
        ax_img.set_facecolor("#1e1e2e")

        row    = hf_ds[int(idx)]
        pil    = row["image"]
        if not isinstance(pil, Image.Image):
            pil = Image.fromarray(pil)
        ax_img.imshow(pil.convert("RGB"))
        ax_img.axis("off")

        # 左上に image_idx を表示
        ax_img.text(
            0.02, 0.97, f"#{idx}",
            transform=ax_img.transAxes,
            color="white", fontsize=7, va="top",
            bbox=dict(facecolor="black", alpha=0.55, pad=1.5, edgecolor="none"),
        )

        # ── テキストエリア ──────────────────────────────────
        ax_txt = fig.add_subplot(inner[1])
        ax_txt.set_facecolor("#13131f")
        ax_txt.axis("off")

        gen_cap = get_generated_caption(df, idx)
        meta    = get_meta(row, col_map)
        orig    = get_orig_text(row)

        y = 0.97
        dy = 1 / (ax_txt.get_window_extent().height or 110)

        # ① 生成キャプション（緑）
        cap_text  = wrap(f"✅ {gen_cap}", 40)
        n_lines   = cap_text.count("\n") + 1
        ax_txt.text(0.03, y, cap_text,
            transform=ax_txt.transAxes,
            color="#a6e3a1", fontsize=7.5, va="top",
            linespacing=1.3)
        y -= n_lines * 8.5 * dy + 0.06

        # 区切り線
        if y > 0.1:
            ax_txt.axhline(y=y + 0.03, color="#45475a", linewidth=0.5)
            y -= 0.07

        # ② メタデータ（青）
        if meta and y > 0.05:
            meta_text = wrap(f"🏷 {meta}", 46)
            n_m       = meta_text.count("\n") + 1
            ax_txt.text(0.03, max(y, 0.02), meta_text,
                transform=ax_txt.transAxes,
                color="#89b4fa", fontsize=6.5, va="top",
                linespacing=1.2)
            y -= n_m * 7.5 * dy + 0.05

        # ③ 元テンプレートキャプション（グレー・比較用）
        if orig and y > 0.02:
            ax_txt.text(0.03, max(y, 0.01), wrap(f"⚠ {orig}", 46),
                transform=ax_txt.transAxes,
                color="#6c7086", fontsize=6.0, va="top",
                linespacing=1.2, style="italic")

    # 余白セルを非表示
    for pos in range(n, nrows * ncols):
        r, c = divmod(pos, ncols)
        ax   = fig.add_subplot(outer[r, c])
        ax.set_visible(False)

    # タイトル
    title = f"WikiArt — BLIP Caption Review  (n={n}, seed={seed})"
    if style_filter:
        title += f"  Style={style_filter}"
    if genre_filter:
        title += f"  Genre={genre_filter}"
    fig.suptitle(title, color="white", fontsize=10, y=0.998)

    plt.tight_layout(rect=[0, 0, 1, 0.997])

    if save_path:
        p = Path(save_path)
        p.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(p, dpi=150, bbox_inches="tight",
                    facecolor=fig.get_facecolor())
        print(f"保存しました: {p}")

    plt.show()

print("show_grid() 定義完了 ✓")


In [ ]:
# ── 実行 ──────────────────────────────────────────────────────
# SEED を変えるたびに別サンプルが表示される
show_grid(df_caps, hf_ds, col_map)


## ⑤ シードを変えて再サンプリング

In [ ]:
# シードをインクリメントしながら確認していく
show_grid(df_caps, hf_ds, col_map, seed=SEED + 1)


## ⑥ スタイル / ジャンルで絞り込んで表示

In [ ]:
# 例: Impressionism だけ表示（STYLE_FILTER を上書き）
show_grid(df_caps, hf_ds, col_map, style_filter="Impressionism", seed=0)


## ⑦ image_idx を直接指定して確認

In [ ]:
def show_single(image_idx: int):
    """1 枚の画像を大きく表示し、生成キャプション・メタデータを並べる。"""
    row = hf_ds[int(image_idx)]
    pil = row["image"]
    if not isinstance(pil, Image.Image):
        pil = Image.fromarray(pil)

    fig, axes = plt.subplots(1, 2, figsize=(10, 5),
                             gridspec_kw={"width_ratios": [1, 1]})
    fig.patch.set_facecolor("#1e1e2e")

    # 左: 画像
    axes[0].imshow(pil.convert("RGB"))
    axes[0].axis("off")
    axes[0].set_title(f"image_idx = {image_idx}", color="white", fontsize=10)

    # 右: テキスト情報
    axes[1].set_facecolor("#13131f")
    axes[1].axis("off")

    gen_cap = get_generated_caption(df_caps, image_idx)
    meta    = get_meta(row, col_map)
    orig    = get_orig_text(row)

    text_block = (
        f"✅ 生成キャプション\n{wrap(gen_cap, 50)}\n\n"
        f"🏷 メタデータ\n{meta}\n\n"
        f"⚠ 元テンプレート\n{wrap(orig, 50)}"
    )
    axes[1].text(
        0.05, 0.95, text_block,
        transform=axes[1].transAxes,
        color="white", fontsize=9, va="top",
        linespacing=1.6,
        bbox=dict(facecolor="#1e1e2e", alpha=0.8, pad=8, edgecolor="#45475a"),
    )

    plt.tight_layout()
    plt.show()


# 使い方: image_idx を直接指定
show_single(0)


## ⑧ キャプション統計（任意）

In [ ]:
# キャプション長の分布などを簡易確認
df_caps["caption_len"] = df_caps["caption"].str.len()

print(df_caps["caption_len"].describe().to_string())
print()

df_caps["caption_len"].hist(bins=40, color="#89b4fa", edgecolor="#1e1e2e")
plt.title("Caption Length Distribution", color="black")
plt.xlabel("Characters")
plt.ylabel("Count")
plt.tight_layout()
plt.show()
